# Protocol 0 — HEP Proxy Validation

Simulates Pred 0.A–0.C from the APGI empirical-prerequisite protocol
(*HEP Proxy Validation: Establishing the Heartbeat-Evoked Potential as a Πⁱ Index*),
updated to **protocol specification v1.1.0**:

* **Pred 0.A** — HEP amplitude correlates with an orthogonal Πⁱ index, 100-trial heartbeat-discrimination d′
  (r > 0.35, two-tailed α = 0.01, N = 115). An independent validation sample (N ≥ 30) must replicate with
  r ≥ 0.25 and a 95 % BCa CI (10 000 resamples) excluding zero.
  *Falsified* only if the upper 95 % bound falls below 0.20 in **each** of the two samples.
* **Pred 0.B** — Physostigmine (ACh) increases HEP ≥ 15 % vs. placebo, jointly with Cohen's d ≥ 0.50 and
  BF₁₀ ≥ 100 (paired crossover, α = 0.05, Cauchy scale 0.707), *contingent on central EEG engagement*
  (≥ 10 % spectral change, 8–30 Hz).
* **Pred 0.C** — aINS BOLD tracks HEP trial-by-trial (within-participant r > 0.30, α = 0.05).
  *Falsified* only if the pooled Fisher-z 90 % interval lies wholly inside ±√0.02 (TOST) with a responsive localizer.

Failing to confirm is **not** falsification: each prediction is classified CONFIRMED, FALSIFIED or INDETERMINATE.
The three results then feed the proxy gate, which reclassifies downstream HEP claims rather than invalidating them wholesale.

> **Note:** All data are synthetic. The generative proxy `HEP₂₅₀₋₄₀₀ = a₀ + a₁·Πⁱ + ε_HEP` (a₁ > 0) is
> illustrative; v1.1 assigns no legacy default to unregistered coefficients.
>
> **EP-0 | Paper 1 | Empirical prerequisite (gate) | Depends on: — | Spec v1.1.0, pre-collection | N = 115 + independent validation N ≥ 30**

### What changed from v1.0

| Item | v1.0 notebook | v1.1 specification |
|---|---|---|
| Primary N | 50 | 115 (r = 0.35, α = 0.01, power 0.90) |
| Replication | 30-participant subsample | Independent sample N ≥ 30, 95 % BCa CI excluding zero |
| Πⁱ index | Heartbeat counting | 100-trial heartbeat **discrimination** d′ |
| Pred 0.B design | Two independent arms, N = 30 | Within-participant crossover (≥ 7-day washout), central-engagement gate, BF₁₀ |
| Pred 0.C falsifier | none | TOST equivalence, bounds ±√0.02, 90 % CI |
| κ | 100 | `null` — uncalibrated ATP/bit bridge, never ignition steepness |

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy import integrate
from scipy.stats import bootstrap, norm, pearsonr, spearmanr, ttest_1samp, ttest_rel
from scipy.stats import t as t_dist

from apgi.core import BETA_SM_DEFAULT, compute_pi_i_eff

In [ ]:
# Load the v1.1 protocol specification so every threshold below is read from
# the registered JSON rather than retyped by hand.
def load_protocol(filename):
    for root in (Path.cwd(), *Path.cwd().parents):
        path = root / "protocols" / filename
        if path.exists():
            return json.loads(path.read_text(encoding="utf-8"))
    raise FileNotFoundError(f"protocols/{filename} not found above {Path.cwd()}")


PROTO = load_protocol("protocol_0_hep_proxy_validation.json")
assert PROTO["version"] == "1.1.0", PROTO["version"]
PRED = {p["id"]: p for p in PROTO["sub_predictions"]}
print(f"{PROTO['protocol_id']}  spec v{PROTO['version']}  status={PROTO['study_status']}")
for pid, p in PRED.items():
    print(f"  {pid:<9} [{p['status']}]  {p['claim'][:78]}")


def verdict(confirmed, falsified=False):
    """v1.1 three-way decision: failing to confirm is not falsification."""
    if confirmed:
        return "CONFIRMED (synthetic)"
    if falsified:
        return "FALSIFIED (synthetic)"
    return "INDETERMINATE (not confirmed; falsification rule not met)"

In [ ]:
def jzs_bf10(t, n, r=0.707):
    """JZS Bayes factor for a one-sample / paired t-test (Rouder et al., 2009).

    Cauchy prior scale r = 0.707, as registered for the v1.1 Bayesian tests.
    The likelihood ratio is formed inside the integral to avoid underflow.
    """
    nu = n - 1

    def integrand(g):
        ratio = ((1 + t**2 / ((1 + n * g * r**2) * nu)) / (1 + t**2 / nu)) ** (-(nu + 1) / 2)
        prior = (2 * np.pi) ** -0.5 * g**-1.5 * np.exp(-1 / (2 * g))
        return (1 + n * g * r**2) ** -0.5 * ratio * prior

    return float(integrate.quad(integrand, 0, np.inf, limit=200)[0])

## 1 — Protocol 0 parameters (v1.1)

Design counts and decision thresholds are read from `protocol_0_hep_proxy_validation.json`.
Generative coefficients marked *illustrative* are synthetic choices, not registered values.

In [ ]:
THR_A = PRED["Pred 0.A"]["confirming_thresholds"]
THR_B = PRED["Pred 0.B"]["confirming_thresholds"]
THR_C = PRED["Pred 0.C"]["confirming_thresholds"]
FAL_C = PRED["Pred 0.C"]["falsifying_thresholds"]

N_PRIMARY = PROTO["participants"]["n_target"]                             # 115
N_VALIDATION = PROTO["participants"]["n_independent_validation_minimum"]  # 30 (independent sample)
N_HB_TRIALS = PROTO["blocks"]["heartbeat_discrimination_trials"]           # 100

APGI = PROTO["apgi_parameters"]
KAPPA = APGI["kappa"]["value"]  # None — uncalibrated; not used for ignition
BETA_SM = BETA_SM_DEFAULT
assert APGI["beta_SM"]["typical_range"][0] <= BETA_SM <= APGI["beta_SM"]["typical_range"][1]

# Illustrative generative values (synthetic; not registered)
A0, A1 = 0.10, 0.80     # HEP intercept and slope on Πⁱ (proxy validity requires A1 > 0)
PI_I_RESTING = 1.0      # population resting Πⁱ
PHYSO_GAIN = 1.25       # cholinergic gain on Πⁱ under physostigmine

print(f"N primary = {N_PRIMARY}, independent validation N ≥ {N_VALIDATION}, "
      f"heartbeat-discrimination trials = {N_HB_TRIALS}")
print(f"κ = {KAPPA} ({APGI['kappa']['status']}), β_SM = {BETA_SM} "
      f"(typical range {APGI['beta_SM']['typical_range']})")
rng = np.random.default_rng(2025)

## 2 — Prediction 0.A: HEP ~ heartbeat-discrimination d′

In [ ]:
def simulate_sample(n, rng):
    """Resting HEP and 100-trial heartbeat-discrimination d′ for n participants."""
    pi_base = PI_I_RESTING * rng.lognormal(0.0, 0.25, n)  # between-participant Πⁱ
    M_hat = rng.uniform(0.0, 0.5, n)
    pi_eff = np.array([compute_pi_i_eff(float(p), BETA_SM, float(m)) for p, m in zip(pi_base, M_hat)])
    hep = A0 + A1 * pi_eff + rng.normal(0, 0.30, n)

    # Discrimination: half synchronous (signal), half asynchronous (noise) trials
    n_sig = n_noise = N_HB_TRIALS // 2
    d_true = np.clip(0.9 * pi_eff + rng.normal(0, 0.35, n), 0, None)
    hits = rng.binomial(n_sig, norm.cdf(d_true / 2))
    fas = rng.binomial(n_noise, norm.cdf(-d_true / 2))
    # log-linear correction keeps d′ finite at 0 / 100 % rates
    d_prime = norm.ppf((hits + 0.5) / (n_sig + 1)) - norm.ppf((fas + 0.5) / (n_noise + 1))
    return hep, d_prime


def r_ci_fisher(r, n, level=0.95):
    z, se = np.arctanh(r), 1 / np.sqrt(n - 3)
    q = norm.ppf(0.5 + level / 2)
    return np.tanh(z - q * se), np.tanh(z + q * se)


hep_primary, d_prime_primary = simulate_sample(N_PRIMARY, rng)
hep_val, d_prime_val = simulate_sample(N_VALIDATION, rng)  # independent, separately consented sample

r_primary, p_primary = pearsonr(hep_primary, d_prime_primary)
rho_primary, _ = spearmanr(hep_primary, d_prime_primary)
ci_primary = r_ci_fisher(r_primary, N_PRIMARY)

r_val, _ = pearsonr(hep_val, d_prime_val)
boot = bootstrap(
    (hep_val, d_prime_val),
    lambda x, y: np.corrcoef(x, y)[0, 1],
    paired=True, vectorized=False, method="BCa",
    n_resamples=THR_A["replication_bootstrap_resamples"],
    confidence_level=THR_A["replication_ci_level"], random_state=1,
)
ci_val = (boot.confidence_interval.low, boot.confidence_interval.high)

primary_ok = r_primary > THR_A["r_primary_strict_minimum"] and p_primary < THR_A["alpha_two_tailed"]
replication_ok = r_val >= THR_A["replication_r_minimum"] and ci_val[0] > 0
confirmed_A = primary_ok and replication_ok
falsified_A = ci_primary[1] < 0.20 and ci_val[1] < 0.20

print(f"Primary     N={N_PRIMARY}: r = {r_primary:.3f} (95% CI {ci_primary[0]:.3f}–{ci_primary[1]:.3f}), "
      f"p = {p_primary:.2e}; Spearman ρ = {rho_primary:.3f}")
print(f"Validation  N={N_VALIDATION}: r = {r_val:.3f} (95% BCa CI {ci_val[0]:.3f}–{ci_val[1]:.3f})")
print(f"Pred 0.A → {verdict(confirmed_A, falsified_A)}")

The validation conjunct is demanding at N = 30, and v1.1 lists its conjunction power as *pending archived
simulation*. The quick Monte-Carlo check below uses a Fisher-z interval as a fast stand-in for BCa, so it
is illustrative only and does not replace the registered simulation.

In [ ]:
N_SIMS = 1000
sim_rng = np.random.default_rng(11)
passes = 0
for _ in range(N_SIMS):
    h, d = simulate_sample(N_VALIDATION, sim_rng)
    r_sim = np.corrcoef(h, d)[0, 1]
    passes += (r_sim >= THR_A["replication_r_minimum"]) and (r_ci_fisher(r_sim, N_VALIDATION)[0] > 0)
print(f"Illustrative replication-conjunct power at N = {N_VALIDATION}: {passes / N_SIMS:.2f}")

## 3 — Prediction 0.B: Physostigmine crossover elevates HEP

Visits 2 and 3 form a counterbalanced drug/placebo crossover in the same participants.
The HEP test is evaluated only in participants with confirmed central engagement.

> The engagement band, direction, normalisation and window are still awaiting prospective lock, and the JSON
> authorises no dose schedule. The |Δ| ≥ 10 % (8–30 Hz) rule below is a placeholder for the locked marker.

In [ ]:
pi_base = PI_I_RESTING * rng.lognormal(0.0, 0.25, N_PRIMARY)
pi_placebo = np.array([compute_pi_i_eff(float(p), BETA_SM, float(m))
                       for p, m in zip(pi_base, rng.uniform(0.0, 0.5, N_PRIMARY))])
pi_physo = np.array([compute_pi_i_eff(float(p) * PHYSO_GAIN, BETA_SM, float(m))
                     for p, m in zip(pi_base, rng.uniform(0.0, 0.5, N_PRIMARY))])
subject_offset = rng.normal(0, 0.15, N_PRIMARY)  # stable participant HEP offset across visits
hep_placebo = A0 + A1 * pi_placebo + subject_offset + rng.normal(0, 0.12, N_PRIMARY)
hep_physo = A0 + A1 * pi_physo + subject_offset + rng.normal(0, 0.12, N_PRIMARY)

# Central engagement: % spectral change in the registered 8–30 Hz range
spectral_change = rng.normal(18, 8, N_PRIMARY)
engaged = np.abs(spectral_change) >= THR_B["central_spectral_change_percent_minimum"]

hp, hd = hep_placebo[engaged], hep_physo[engaged]
n_eng = int(engaged.sum())
diff = hd - hp
pct_increase = (hd.mean() - hp.mean()) / hp.mean() * 100
cohens_dz = diff.mean() / diff.std(ddof=1)
t_b, p_b = ttest_rel(hd, hp)
bf10_b = jzs_bf10(t_b, n_eng)

confirmed_B = (
    pct_increase >= THR_B["hep_increase_percent_minimum"]
    and cohens_dz >= THR_B["cohens_d_minimum"]
    and bf10_b >= THR_B["bf10_minimum"]
    and p_b < THR_B["alpha"]
)
print(f"Central engagement: {n_eng}/{N_PRIMARY} participants")
print(f"HEP placebo {hp.mean():.3f} ± {hp.std():.3f}   physostigmine {hd.mean():.3f} ± {hd.std():.3f}")
print(f"Increase {pct_increase:.1f}%   Cohen's d_z = {cohens_dz:.2f}   t({n_eng - 1}) = {t_b:.2f}   "
      f"BF₁₀ = {bf10_b:.3g}")
print(f"Pred 0.B → {verdict(confirmed_B)}")

## 4 — Prediction 0.C: aINS BOLD ~ HEP trial-by-trial

Pooled within-participant association on the Fisher-z scale. The per-participant trial count is
unregistered in v1.1, so 60 trials is illustrative.

In [ ]:
N_TRIALS_FMRI = 60  # illustrative; not a registered count

z_within = []
for _ in range(N_PRIMARY):
    pi_i_s = max(0.2, rng.normal(PI_I_RESTING, 0.2))
    pi_eff_s = np.array([compute_pi_i_eff(pi_i_s, BETA_SM, float(m))
                         for m in rng.uniform(0.0, 0.5, N_TRIALS_FMRI)])
    hep_s = A0 + A1 * pi_eff_s + rng.normal(0, 0.10, N_TRIALS_FMRI)
    bold_ains = 0.55 * hep_s + rng.normal(0, 0.20, N_TRIALS_FMRI)  # after HRV/respiration/global-signal control
    z_within.append(np.arctanh(pearsonr(hep_s, bold_ains)[0]))
z_within = np.array(z_within)
r_within = np.tanh(z_within)

z_mean, z_se = z_within.mean(), z_within.std(ddof=1) / np.sqrt(N_PRIMARY)
q90 = t_dist.ppf(0.5 + FAL_C["ci_level"] / 2, N_PRIMARY - 1)
ci90_r = np.tanh([z_mean - q90 * z_se, z_mean + q90 * z_se])
r_pooled = np.tanh(z_mean)
_, p_c = ttest_1samp(z_within, 0.0)

lo_eq, hi_eq = FAL_C["equivalence_bounds"]  # ±√0.02
localizer_responsive = True                 # synthetic localizer passes
confirmed_C = r_pooled > THR_C["within_participant_r_strict_minimum"] and p_c < THR_C["alpha"]
falsified_C = localizer_responsive and lo_eq < ci90_r[0] and ci90_r[1] < hi_eq

print(f"Pooled within-participant r = {r_pooled:.3f} (90% CI {ci90_r[0]:.3f}–{ci90_r[1]:.3f}), p = {p_c:.2e}")
print(f"Equivalence bounds ±√0.02 = ±{hi_eq:.4f}")
print(f"Pred 0.C → {verdict(confirmed_C, falsified_C)}")

## 5 — Proxy gate: downstream reclassification

v1.1 replaces the all-or-nothing gate with targeted reclassification. Unconfirmed criteria demote the
affected downstream claims to exploratory; inadmissible data are not counted as failed tests.

In [ ]:
gate = PROTO["proxy_gate"]
met = {"A": confirmed_A, "B": confirmed_B, "C": confirmed_C}
unmet = [k for k, ok in met.items() if not ok]

if not unmet:
    outcome = gate["full_retention"]
elif len(unmet) == 1:
    outcome = gate[f"{unmet[0]}_only_not_met"]
else:
    outcome = gate["two_or_more_not_met"]
print("Criteria met:", {f"0.{k}": bool(ok) for k, ok in met.items()})
print("Gate outcome:", outcome)
print("Ranked fallback proxies:", ", ".join(gate["fallbacks"]))

## 6 — Summary visualisation (Pred 0.A–0.C)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

# Pred 0.A: HEP vs discrimination d′
ax = axes[0]
ax.scatter(d_prime_primary, hep_primary, alpha=0.55, color="#2166ac", s=22, label=f"Primary (N={N_PRIMARY})")
ax.scatter(d_prime_val, hep_val, alpha=0.7, color="#9966FF", s=22, marker="^",
           label=f"Validation (N={N_VALIDATION})")
m, b = np.polyfit(d_prime_primary, hep_primary, 1)
xs = np.linspace(d_prime_primary.min(), d_prime_primary.max(), 50)
ax.plot(xs, m * xs + b, color="#2166ac", lw=1.5)
ax.set_xlabel("Heartbeat-discrimination d′ (100 trials)")
ax.set_ylabel("HEP amplitude (a.u.)")
ax.set_title(f"Pred 0.A  r = {r_primary:.2f} (primary)\nr = {r_val:.2f}, BCa CI [{ci_val[0]:.2f}, {ci_val[1]:.2f}]")
ax.legend(fontsize=8)
ax.spines[["top", "right"]].set_visible(False)

# Pred 0.B: paired crossover
ax = axes[1]
for a, c in zip(hp, hd):
    ax.plot([0, 1], [a, c], color="#BBBBBB", lw=0.5, alpha=0.5)
ax.errorbar([0, 1], [hp.mean(), hd.mean()],
            yerr=[hp.std() / np.sqrt(n_eng), hd.std() / np.sqrt(n_eng)],
            fmt="o-", color="#2166ac", lw=2, capsize=5)
ax.set_xticks([0, 1], ["Placebo", "Physostigmine"])
ax.set_xlim(-0.4, 1.4)
ax.set_ylabel("HEP amplitude (a.u.)")
ax.set_title(f"Pred 0.B  +{pct_increase:.1f}%  d_z = {cohens_dz:.2f}\nBF₁₀ = {bf10_b:.2g}  (engaged N = {n_eng})")
ax.spines[["top", "right"]].set_visible(False)

# Pred 0.C: within-participant r with equivalence band
ax = axes[2]
ax.hist(r_within, bins=16, color="#00CC99", alpha=0.85, edgecolor="white")
ax.axvspan(lo_eq, hi_eq, color="#AAAAAA", alpha=0.3, label="equivalence ±√0.02")
ax.axvline(THR_C["within_participant_r_strict_minimum"], ls="--", lw=1.5, color="k", label="threshold r = 0.30")
ax.axvline(r_pooled, ls="-", lw=2, color="#2166ac", label=f"pooled r = {r_pooled:.2f}")
ax.set_xlabel("Within-participant r(HEP, aINS BOLD)")
ax.set_ylabel("Participants")
ax.set_title("Pred 0.C — aINS BOLD ~ HEP")
ax.legend(fontsize=7)
ax.spines[["top", "right"]].set_visible(False)

fig.suptitle("Protocol 0 — HEP Proxy Validation (Pred 0.A–0.C) · spec v1.1.0 · synthetic data", fontsize=12)
plt.tight_layout()
plt.show()